In [9]:
from dotenv import load_dotenv
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from pymilvus import MilvusClient

from app.core.config import settings
#=====================1.准备工作===================
load_dotenv()

#切分文档函数
def prepare_doc():
    with open("../resources/评估.md","r",encoding="utf-8") as f:
        md_doc = f.read()

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#","h1"),("##","h2"),("###","h3")],
        strip_headers=False
    )
    return splitter.split_text(md_doc)

#向量模型：
embedding = DashScopeEmbeddings(model='text-embedding-v4')

#=====================2.Milvus-Entity操作===================
client = MilvusClient(uri = settings.rag.milvus_url)

#获取文档片段列表
doc_list = prepare_doc()
#稠密向量列表
doc_vector_list = embedding.embed_documents([doc.page_content for doc in doc_list])
#组装成对应collection结构的字典对象
entity_list = [{
   "id": i+1,
    "h1": doc.metadata.get("h1", " "),
    "h2": doc.metadata.get("h2", " "),
    "h3": doc.metadata.get("h3", " "),
    "text": doc.page_content,
    "dense": doc_vector_list[i]
}for i,doc in enumerate(doc_list)]
#保存到集合中
res = client.insert(collection_name="my_1",data=entity_list)
print(f"> 添加文档到Milvus的my_1的集合中:{res}")

# 2. 修改文档
entity1 = entity_list[0]
entity1['h2'] = 'hello'
entity2 = entity_list[1]
entity2['id'] = 100
res = client.upsert(collection_name="my_1",data = [entity1,entity2])
print(res)

# 3. 删除数据 :client.delete()
res = client.delete(collection_name="my_1",ids=[1,2,100])
print(res)

> 添加文档到Milvus的my_1的集合中:{'insert_count': 21, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]}
